# Used Car Price Prediction: Baseline Model

## Project Objective

The objective of this project is to predict the selling price of used cars based on their characteristics, such as kilometers driven, fuel type, seller type, transmission and ownership history.

This notebook builds and evaluates the **baseline** of a basic machine learning regression workflow:

- Data preparation and feature selection
- Train-test splitting
- Feature preprocessing
- Pipeline with a log-transformed target
- 5-fold cross-validation
- Dummy baseline vs Linear Regression comparison

**Evaluation protocol:** all model comparison is done with 5-fold cross-validation on the training set. The test set is held out and will be used exactly once, at the end of the project, for the final chosen model.

# 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

file_path = Path('..') / 'Data' / 'car_data_cleaned.csv'
df = pd.read_csv(file_path)
display(df.head())

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,car_age
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,19
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,19
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,14
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,9
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,12


In [2]:
print(df.duplicated().sum())
df = df.drop(columns=['year'])

0


### Duplicate Check

The dataset contains **0 duplicate rows**, so no exact duplicate records were found.

This reduces the risk of identical records appearing in both the training and testing datasets, which could otherwise lead to data leakage and an overly optimistic evaluation of model performance.

### Dropping the Year Column

The `year` column was dropped because the dataset already contains a derived feature, `car_age`, calculated as `car_age = 2026 - year`.

Since both columns represent the same underlying information, keeping both would introduce redundant features. Therefore `year` was removed and `car_age` is kept as the representative feature for vehicle age.

# 2. Splitting the Dataset into Training and Test Sets

In [3]:
X = df.drop(columns=['selling_price'])  # Features
y = df['selling_price']  # Target values

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_test.shape)
print(X_train.shape)
print(y_test.shape)
print(y_train.shape)

(716, 7)
(2861, 7)
(716,)
(2861,)


### Splitting the Dataset into Training and Testing Sets

The dataset was divided into **features (`X`)** and the **target variable (`y`)**.

- **Features (`X`):** all columns except `selling_price`, used by the model to make predictions.
- **Target (`y`):** the `selling_price` column, which the model aims to predict.

The dataset was then split using `train_test_split()` with the following configuration:

- **Training set:** 80% (2,861 observations)
- **Testing set:** 20% (716 observations)
- **Random state:** 42, ensuring a reproducible split.

| Dataset | Observations | Features / Values |
|---|---:|---:|
| `X_train` | 2,861 | 7 features |
| `X_test` | 716 | 7 features |
| `y_train` | 2,861 | 1 target |
| `y_test` | 716 | 1 target |

**Purpose of the split:**

The training set is used to build and compare models (with cross-validation). The test set is kept untouched so that the final model can be evaluated once on truly unseen data, which shows how well it generalizes.

# 3. Feature Selection and Preprocessing

### Excluding the Name Column from Baseline Features

The `name` column contains the full names and variants of individual car models. It was excluded from the baseline model for the following reasons:

- The column contains numerous unique car names and variants, resulting in high cardinality.
- Applying One-Hot Encoding directly to this column would generate a very large number of additional features.

Instead, the baseline uses more general vehicle characteristics: fuel type, transmission, seller type, ownership history, kilometers driven and car age.

`name` stays inside `X_train`, but the `ColumnTransformer` below only selects the listed columns and drops everything else by default, so `name` never reaches the model. A lower-cardinality feature extracted from it (such as brand) is the natural next step.

### Feature Preprocessing

**1. Categorical features: One-Hot Encoding**

The following columns contain categorical values and need to be converted into numerical representations:

- `fuel`
- `seller_type`
- `transmission`
- `owner`

**2. Numerical features: Standard Scaling**

The following columns contain numerical values and are standardized using `StandardScaler`:

- `km_driven`
- `car_age`

**Reason for preprocessing:**

One-Hot Encoding converts categorical features into binary numerical columns, while `StandardScaler` puts numerical features on a common scale.

`handle_unknown='ignore'` prevents errors when a validation or test row contains a category that was not seen during training.

In [5]:
categorical_features = ['fuel', 'seller_type', 'transmission', 'owner']
numerical_features = ['km_driven', 'car_age']

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numerical', StandardScaler(), numerical_features)
    ]
)

**Data Leakage Prevention:**

The preprocessor lives inside the model pipeline, so it is always fitted on training data only. The `OneHotEncoder` learns its categories and the `StandardScaler` learns its mean and standard deviation without seeing validation or test rows.

During cross-validation, the whole pipeline (encoder, scaler and model) is cloned and refit on each training fold, so the validation fold never influences preprocessing. The held-out test set is not touched at all.

This prevents data leakage and keeps every evaluation fair.

# 4. Pipeline and Log-Transformed Target

In [6]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

cv_pipeline = TransformedTargetRegressor(
    regressor=model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

### Log Transformation of the Target Variable

The `selling_price` column is right-skewed: a small number of high-priced vehicles stretch the upper range of prices.

To reduce the effect of this skewness, the model is trained on `np.log1p(selling_price)`. This is done with `TransformedTargetRegressor`, which:

- applies `log1p` to the target when fitting, separately inside every CV fold
- applies `expm1` to the predictions automatically, so all metrics are reported in the original price scale (₹)

Note: converting log-scale predictions back with `expm1` gives an estimate closer to the median price than the mean price for each car.

# 5. Cross-Validation Setup

All models are compared with **5-fold cross-validation on the training set only**.

- `KFold(n_splits=5, shuffle=True, random_state=42)` is used for every model, so all models are evaluated on identical folds.
- The same three metrics are used everywhere: MAE, RMSE and R² (all in the original ₹ scale).
- The `test_*` keys returned by `cross_validate` are **validation-fold** scores, not scores on the held-out test set.
- The held-out test set is not used in this notebook.

In [7]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

# 6. Dummy Baseline (5-Fold CV)

The dummy baseline ignores all features and predicts the **median of the training fold** for every car.

**Purpose:** it sets the minimum bar. A real model is only useful if it clearly beats this trivial prediction on the same folds and the same metrics.

The median is used instead of the mean because the price distribution is right-skewed.

In [8]:
dummy_results = cross_validate(
    estimator=DummyRegressor(strategy='median'),
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

# 7. Linear Regression: 5-Fold Cross-Validation

In [9]:
cv_results = cross_validate(
    estimator=cv_pipeline,
    X=X_train,
    y=y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True
)

cv_summary = pd.DataFrame({
    'Fold': range(1, kf.get_n_splits() + 1),

    'Train_MAE': -cv_results['train_MAE'],
    'Val_MAE': -cv_results['test_MAE'],

    'Train_RMSE': -cv_results['train_RMSE'],
    'Val_RMSE': -cv_results['test_RMSE'],

    'Train_R2': cv_results['train_R2'],
    'Val_R2': cv_results['test_R2']
})

display(cv_summary)

summary = cv_summary.drop(columns='Fold').agg(['mean', 'std']).T
display(summary)

# Gaps: for all three, a large positive value suggests overfitting
mae_gap = cv_summary['Val_MAE'].mean() - cv_summary['Train_MAE'].mean()
rmse_gap = cv_summary['Val_RMSE'].mean() - cv_summary['Train_RMSE'].mean()
r2_gap = cv_summary['Train_R2'].mean() - cv_summary['Val_R2'].mean()

print(f'MAE gap  (Val - Train): {mae_gap:,.0f}')
print(f'RMSE gap (Val - Train): {rmse_gap:,.0f}')
print(f'R2 gap   (Train - Val): {r2_gap:.4f}')

,Fold,Train_MAE,Val_MAE,Train_RMSE,Val_RMSE,Train_R2,Val_R2
0,1,168673.721699,175328.571804,327556.937109,446796.504594,0.528832,0.349365
1,2,172055.564039,164637.281466,364373.938338,303167.204194,0.474460,0.556542
2,3,171125.176806,167955.020613,351370.678309,365322.079262,0.491032,0.460047
3,4,173542.833629,158499.820794,373923.556598,265363.371557,0.462086,0.602547
4,5,165380.164175,191035.205129,355636.211209,375843.581337,0.461339,0.488388


,mean,std
Train_MAE,170155.492069,3202.500771
Val_MAE,171491.179961,12499.502555
Train_RMSE,354572.264313,17405.138370
Val_RMSE,351298.548189,70036.523705
Train_R2,0.483550,0.028032
Val_R2,0.491378,0.097164


MAE gap  (Val - Train): 1,336
RMSE gap (Val - Train): -3,274
R2 gap   (Train - Val): -0.0078


### 5-Fold CV Results (Linear Regression, log-target)

| Metric | Train (mean) | Validation (mean ± std) |
|---|---:|---:|
| MAE | ₹1.70L | ₹1.71L ± 0.12L |
| RMSE | ₹3.55L | ₹3.51L ± 0.70L |
| R² | 0.484 | 0.491 ± 0.097 |

**Gaps** (all defined so that a large positive value means overfitting): MAE gap ≈ +₹1,336, RMSE gap ≈ -₹3,274, R² gap ≈ -0.008.

**Interpretation:**

- **No overfitting:** train and validation scores are nearly identical, and all three gaps are close to zero.
- **Underfitting (high bias):** both R² values are about 0.49, so the model explains only about half of the price variation. More data or regularization will not fix this; the model needs better features or a more flexible learner.
- **RMSE is less stable than MAE:** RMSE varies about 20% across folds, MAE about 7%. RMSE squares errors, so a few very expensive cars landing in one fold swing it.

**Purpose:** CV is used to compare models during development. The test set is reserved for one final check.

# 8. Dummy Baseline vs Linear Regression

In [10]:
def summarize(name, res):
    return {
        'Model': name,
        'Val_MAE': -res['test_MAE'].mean(),
        'Val_MAE_std': res['test_MAE'].std(ddof=1),
        'Val_RMSE': -res['test_RMSE'].mean(),
        'Val_RMSE_std': res['test_RMSE'].std(ddof=1),
        'Val_R2': res['test_R2'].mean(),
        'Val_R2_std': res['test_R2'].std(ddof=1),
    }

comparison = pd.DataFrame([
    summarize('Dummy (median)', dummy_results),
    summarize('Linear Regression (log target)', cv_results),
])
display(comparison.round(3))

mae_reduction = (1 - comparison.loc[1, 'Val_MAE'] / comparison.loc[0, 'Val_MAE']) * 100
print(f'Validation MAE reduction vs dummy: {mae_reduction:.2f}%')

,Model,Val_MAE,Val_MAE_std,Val_RMSE,Val_RMSE_std,Val_R2,Val_R2_std
0,Dummy (median),273095.826,21351.390,505761.755,56746.683,-0.063,0.023
1,Linear Regression (log target),171491.180,12499.503,351298.548,70036.524,0.491,0.097


Validation MAE reduction vs dummy: 37.20%


### Observations

- The dummy predicts the training median for every car, so its R² sits near zero by construction.
- Linear Regression's validation R² of about 0.49 and MAE of about ₹1.71L show that the car features carry real signal beyond the median price.
- The printed MAE reduction versus the dummy is the headline improvement of this stage.
- Because every number comes from the same 5 folds on the training set, this comparison is fair and the test set stays untouched.

# 9. Conclusion and Next Steps

**Baseline conclusion**

- Linear Regression with a log-transformed target clearly beats the dummy baseline.
- Train and validation scores match, so the model is not overfitting; with R² of about 0.49 it is **underfitting**.
- The `name` column (brand and model) was left out, and it is likely the strongest missing price signal.